In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\E706149\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\E706149\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import openmeteo_requests

import requests_cache
import pandas as pd
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.37927498244628,
	"longitude": 4.9006218076530725,
	"hourly": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation_probability", "precipitation", "cloud_cover", "wind_speed_10m","shortwave_radiation","diffuse_radiation", "direct_normal_irradiance",],
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
#print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
#print(f"Elevation {response.Elevation()} m asl")
#print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
#print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(5).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(6).ValuesAsNumpy()
hourly_GHI = hourly.Variables(7).ValuesAsNumpy()
hourly_DHI = hourly.Variables(8).ValuesAsNumpy()
hourly_DNI = hourly.Variables(9).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["GHI"] = hourly_GHI
hourly_data["DHI"] = hourly_DHI
hourly_data["DNI"] = hourly_DNI

hourly_dataframe = pd.DataFrame(data = hourly_data)
#hourly_dataframe

In [2]:
import pandas as pd

# Assuming your DataFrame is named `hourly_dataframe` and has already been loaded.

# Ensure the 'date' column is in datetime format
hourly_dataframe['date'] = pd.to_datetime(hourly_dataframe['date'])

# Set 'date' as the index
hourly_dataframe.set_index('date', inplace=True)

# Resample to 15-minute intervals, forward filling other columns
resampled_df = hourly_dataframe.resample('15T').ffill()

# Adjust precipitation by dividing by 4
resampled_df['precipitation'] = resampled_df['precipitation'] / 4

# Reset index if you want the 'date' to be a column again
resampled_df.reset_index(inplace=True)

# Display the resampled DataFrame
print(resampled_df)

                         date  temperature_2m  relative_humidity_2m  \
0   2025-09-05 00:00:00+00:00          16.437                  85.0   
1   2025-09-05 00:15:00+00:00          16.437                  85.0   
2   2025-09-05 00:30:00+00:00          16.437                  85.0   
3   2025-09-05 00:45:00+00:00          16.437                  85.0   
4   2025-09-05 01:00:00+00:00          16.837                  79.0   
..                        ...             ...                   ...   
664 2025-09-11 22:00:00+00:00          15.950                  70.0   
665 2025-09-11 22:15:00+00:00          15.950                  70.0   
666 2025-09-11 22:30:00+00:00          15.950                  70.0   
667 2025-09-11 22:45:00+00:00          15.950                  70.0   
668 2025-09-11 23:00:00+00:00          15.600                  72.0   

     apparent_temperature  precipitation_probability  precipitation  \
0               15.657528                        0.0            0.0   
1    

C:\Users\stayr\AppData\Local\Temp\ipykernel_3200\200160861.py:12: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_df = hourly_dataframe.resample('15T').ffill()


In [46]:

# Convert to list of dictionaries
json_data = resampled_df.to_dict(orient="records")

#print(json_data)


In [47]:
json_data
hourly_dataframe.to_json("weather_AMS_data_forecast.json", orient="records", date_format="iso")
